<a href="https://colab.research.google.com/github/OscarGalloGarcia/Msc-artificial-neural-networks/blob/main/P2_Adaline_RedesNeuronalesArtificiales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica 2: Adaline

Oscar Alberto Gallo García
Msc en Robótica e Inteligencia Artificial



In [71]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Clase adaline

In [72]:
class adaline:
  def __init__(self, n_inputs, learning_rate, stop, shuffle):
    self.rgen = np.random.RandomState(1) # Semilla para datos random
    self.w = self.rgen.normal(loc=0.0, scale=0.01, size=n_inputs) #pesos iniciales
    self.b = self.rgen.normal(loc=0.0, scale=0.01) # bias inicial como escalar flotante

    self.learning_rate = learning_rate # eta -> learning rate
    self.stop = stop # criterio de paro
    self.shuffle = shuffle #mezclar patrones al inicio

    self.mse = [] #mean square error por epoca

  def net_input(self, X):
    return np.dot(X, self.w) + self.b # v

  def activation(self, v):
    return v #funcion de activacion identidad (salida lineal)

  def train(self, X, y_d, epochs):
    n_features = X.shape #numero de caracteristicas
    self.mse = []

    for epoch in range(epochs):
      index = np.arange(n_features)
      if self.shuffle:
        self.rgen.shuffle(index)

      # Stochastic Gradient Descent (SDG)
      for k in index:
        x_k = X[k]
        v = np.dot(x_k, self.w) + self.b
        y = self.activation(v)

        #error
        e = y_d[k] - y

        #actualizacion de pesos
        self.w += self.learning_rate * (y_d[k] - y) * x_k
        self.b += self.learning_rate * (y_d[k] - y)

      #MSE
      e_epoch = y_d - self.activation(self.net_input(X))
      mse = (e_epoch ** 2).sum() / 2.0
      self.mse.append(np.mean(mse))

      # criterio de paro
      if not np.isfinite(self.mse[-1]):
        break
      if epoch > 0 and abs(self.mse[-1] - self.mse[-2]) < self.stop:
        break
    return self

  def predict(self, X):
    # Devuelve las predicciones de clase (1 o -1) usando un umbral en 0.0
    return np.where(self.activation(self.net_input(X)) >= 0.0, 1, -1)

Ingesta de datos

In [74]:
!wget 'https://github.com/OscarGalloGarcia/Msc-artificial-neural-networks/raw/main/datasets/ensaantro2022_entrega_w.csv.csv/ensaantro2022_entrega_w.csv'



--2026-10-13 01:42:15--  https://github.com/OscarGalloGarcia/Msc-artificial-neural-networks/raw/main/datasets/ensaantro2022_entrega_w.csv.csv/ensaantro2022_entrega_w.csv
Resolving github.com (github.com)... 20.29.134.23
Connecting to github.com (github.com)|20.29.134.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/OscarGalloGarcia/Msc-artificial-neural-networks/main/datasets/ensaantro2022_entrega_w.csv.csv/ensaantro2022_entrega_w.csv [following]
--2026-10-13 01:42:15--  https://raw.githubusercontent.com/OscarGalloGarcia/Msc-artificial-neural-networks/main/datasets/ensaantro2022_entrega_w.csv.csv/ensaantro2022_entrega_w.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5818250 (5.5M) [text/

Lectura de datos

In [75]:
dataset = pd.read_csv('ensaantro2022_entrega_w.csv')
dataset.head()

,FOLIO_I,FOLIO_INT,t_hora,t_min,t_sumai,t_sumaf,hora_ini_1,fecha_ini_1,hora_fin_1,fecha_fin_1,...,comentario,tiempo,hora_fin,fecha_fin,completa,otroent,ponde_f,estrato,est_sel,upm
0,2022_01001001,2022_01001001_01,20,31,1228,1231,20:28:39,09/11/2022,20:31:33,09/11/2022,...,,3,20:31:33,09/11/2022,1,,999.761286,3,13,0100100011320
1,2022_01001001,2022_01001001_02,20,28,1226,1228,20:26:30,09/11/2022,20:28:37,09/11/2022,...,,2,20:28:37,09/11/2022,1,,1546.785919,3,13,0100100011320
2,2022_01001003,2022_01001003_02,20,36,1231,1236,20:31:35,09/11/2022,20:36:37,09/11/2022,...,,5,20:36:37,09/11/2022,1,,2926.133592,3,13,0100100011320
3,2022_01001003,2022_01001003_03,19,29,1080,1169,18:00:46,10/11/2022,19:29:56,10/11/2022,...,"ta correcta de la 3° es de 89/43 , se coloca 8...",89,19:29:56,10/11/2022,1,,1819.859020,3,13,0100100011320
4,2022_01001003,2022_01001003_06,20,37,1236,1237,20:36:38,09/11/2022,20:37:59,09/11/2022,...,,1,20:37:59,09/11/2022,1,,4640.357758,3,13,0100100011320


In [76]:
# Total de columnas:
num_columnas_len = len(dataset.columns)
print(f"Número de columnas: {num_columnas_len}")

Número de columnas: 98


In [77]:
dataset[['an01_1', 'an04_1', 'h0303']].dtypes

,0
an01_1,object
an04_1,object
h0303,int64


In [78]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15073 entries, 0 to 15072
Data columns (total 98 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FOLIO_I      15073 non-null  object 
 1   FOLIO_INT    15073 non-null  object 
 2   t_hora       15073 non-null  int64  
 3   t_min        15073 non-null  int64  
 4   t_sumai      15073 non-null  int64  
 5   t_sumaf      15073 non-null  int64  
 6   hora_ini_1   15073 non-null  object 
 7   fecha_ini_1  15073 non-null  object 
 8   hora_fin_1   15073 non-null  object 
 9   fecha_fin_1  15073 non-null  object 
 10  tiempo1      15073 non-null  int64  
 11  resultado_1  15073 non-null  int64  
 12  hora_ini_2   15073 non-null  object 
 13  fecha_ini_2  15073 non-null  object 
 14  hora_fin_2   15073 non-null  object 
 15  fecha_fin_2  15073 non-null  object 
 16  tiempo2      15073 non-null  int64  
 17  resultado_2  15073 non-null  int64  
 18  hora_ini_3   15073 non-null  object 
 19  fech

In [79]:
#Hacer una copia del dataset que sera con el que voy a trabajar
df = dataset.copy()

Limpieza de datos

In [80]:
# Las columnas de peso y estatura estan revueltas con nomenclaturas extrañas
# Lo primero es crear columnas limpias que tengan esa data

#       Edad     embarazo  peso     estatura
cols = ['h0303', 'an06', 'an01_1', 'an04_1']
# Hacerlos numericos
df = df[cols].apply(pd.to_numeric, errors='coerce')

datos = df[
      df['an01_1'].between(2, 250)   # peso válido (quita 222.22)
    & df['an04_1'].between(40, 230)  # estatura válida (quita 222.20)
    & df['h0303'].between(20, 110)   # edad -> adultos
    & ~df['an06'].isin([1, 2, 3])    # sin embarazo ni lactancia
].copy()

datos['altura'] = datos['an04_1'] / 100 #metros
datos['peso']   = datos['an01_1']   # Kg



Calcular salida deseada a partir de la formula de IMC

In [81]:
# Formula IMC
datos['imc']    = datos['peso'] / datos['altura'] ** 2
datos['d']      = np.where(datos['imc'] >= 25, 1, -1)

In [82]:
datos = datos[['altura', 'peso', 'imc', 'd']].reset_index(drop=True)
datos.head()

,altura,peso,imc,d
0,1.416,69.30,34.562626,1
1,1.568,110.35,44.882829,1
2,1.773,104.40,33.211082,1
3,1.664,77.00,27.808917,1
4,1.734,81.50,27.105625,1


Entrada y salda

In [83]:
x = np.asanyarray(datos.drop(columns=['imc', 'd']))

y = np.asanyarray(datos[['d']])

Particionar el dataset 70% train, 20% test

In [84]:
train = datos.sample(frac=0.8, random_state=42)
test = datos.drop(train.index)

Crear instancia del adaline

In [85]:
_, n_inputs = x.shape

In [86]:
a = adaline(n_inputs=n_inputs,
            learning_rate=0.01,
            shuffle=True,
            stop=1e-6)

Entrenamiento

In [87]:
a.train(X=x, y_d=y, epochs=100)

TypeError: arange: scalar arguments expected instead of a tuple.